# Logical Planning: A STRIPS-Style Warehouse Robot

Notebook for the *Logical Reasoning for Planning* lab. An LLM (Claude) helped generate the
planner implementation, as the lab asks, after the problem was specified by hand below.

## Task 0 - Writing down the planning problem

Three locations $A, B, C$. The robot and the package both start at $A$; the goal is for the
package to reach $C$.

$$I = \{\text{At(Robot}, A), \text{At(Package}, A)\}, \qquad G = \{\text{At(Package}, C)\}$$

Actions, with their preconditions and effects:

| Action | Preconditions | Effects |
|---|---|---|
| Move(X, Y) *(for connected X, Y: A-B, B-C)* | At(Robot, X) | ¬At(Robot, X), At(Robot, Y) |
| PickUp(Package, X) | At(Robot, X), At(Package, X) | ¬At(Package, X), Holding(Package) |
| Drop(Package, X) | At(Robot, X), Holding(Package) | ¬Holding(Package), At(Package, X) |

**Is `PickUp(Package, A)` applicable in $I$?** Yes - both of its preconditions,
At(Robot, A) and At(Package, A), hold in $I$.
**Is `Drop(Package, C)` applicable in $I$?** No - neither At(Robot, C) nor
Holding(Package) holds yet, so neither precondition is satisfied.

## Task 1 - A plan worked out by hand

Since the package starts in the same place as the robot, it should be picked up *before*
any moving happens, then dropped once both have reached $C$:

| State | Facts true | Action applied next |
|---|---|---|
| $S_0$ | At(Robot,A), At(Package,A) | PickUp(Package,A) |
| $S_1$ | At(Robot,A), Holding(Package) | Move(A,B) |
| $S_2$ | At(Robot,B), Holding(Package) | Move(B,C) |
| $S_3$ | At(Robot,C), Holding(Package) | Drop(Package,C) |
| $S_4$ | At(Robot,C), At(Package,C) | - goal reached - |

$S_4 \models G$, so this four-action plan is valid. It will be used below to check the
generated program's output rather than trusting it blindly.

## Task 2 - Letting an LLM implement the planner

Prompt used (the lab's own template, filled in for this problem):

> *Implement a simple STRIPS-style planning agent in Python. Represent a state as a set of
> propositions. Each action has a name, positive preconditions, negative preconditions,
> positive effects and negative effects. An action is applicable when every one of its
> preconditions holds in the current state. Applying an action removes its negative effects
> from the state and adds its positive effects. Use breadth-first search over the state
> space to find a sequence of actions reaching a given goal. Report "no plan found" if the
> goal is unreachable, otherwise print the plan and the state after each step.*

The code below is the result, read through afterwards to confirm it does what the prompt
specifies (see the table in Task 4).

In [1]:
class Action:
    def __init__(self, name, pre_pos, pre_neg, eff_pos, eff_neg):
        self.name = name
        self.pre_pos = frozenset(pre_pos)
        self.pre_neg = frozenset(pre_neg)
        self.eff_pos = frozenset(eff_pos)
        self.eff_neg = frozenset(eff_neg)

    def applicable(self, state):
        return self.pre_pos <= state and not (self.pre_neg & state)

    def apply(self, state):
        return (state - self.eff_neg) | self.eff_pos

    def __repr__(self):
        return self.name

In [2]:
def warehouse_actions():
    actions = []
    for x, y in [("A", "B"), ("B", "A"), ("B", "C"), ("C", "B")]:
        actions.append(Action(
            f"Move({x},{y})",
            pre_pos={f"At(Robot,{x})"}, pre_neg=set(),
            eff_pos={f"At(Robot,{y})"}, eff_neg={f"At(Robot,{x})"},
        ))
    for loc in ("A", "B", "C"):
        actions.append(Action(
            f"PickUp(Package,{loc})",
            pre_pos={f"At(Robot,{loc})", f"At(Package,{loc})"}, pre_neg=set(),
            eff_pos={"Holding(Package)"}, eff_neg={f"At(Package,{loc})"},
        ))
        actions.append(Action(
            f"Drop(Package,{loc})",
            pre_pos={f"At(Robot,{loc})", "Holding(Package)"}, pre_neg=set(),
            eff_pos={f"At(Package,{loc})"}, eff_neg={"Holding(Package)"},
        ))
    return actions

ACTIONS = warehouse_actions()
print(len(ACTIONS), "actions available")
for a in ACTIONS:
    print(" ", a)

10 actions available
  Move(A,B)
  Move(B,A)
  Move(B,C)
  Move(C,B)
  PickUp(Package,A)
  Drop(Package,A)
  PickUp(Package,B)
  Drop(Package,B)
  PickUp(Package,C)
  Drop(Package,C)


In [3]:
from collections import deque

def plan(initial, goal, actions):
    initial = frozenset(initial)
    goal = frozenset(goal)
    frontier = deque([initial])
    came_from = {initial: None}   # state -> (previous state, action)
    trace = [initial]

    while frontier:
        state = frontier.popleft()
        if goal <= state:
            steps, cur = [], state
            while came_from[cur] is not None:
                cur, action = came_from[cur]
                steps.append(action)
            return list(reversed(steps)), _reconstruct_states(initial, list(reversed(steps)))
        for a in actions:
            if a.applicable(state):
                nxt = a.apply(state)
                if nxt not in came_from:
                    came_from[nxt] = (state, a)
                    frontier.append(nxt)
    return None, None

def _reconstruct_states(initial, steps):
    states = [initial]
    cur = initial
    for a in steps:
        cur = a.apply(cur)
        states.append(cur)
    return states

In [4]:
INITIAL = {"At(Robot,A)", "At(Package,A)"}
GOAL = {"At(Package,C)"}

steps, states = plan(INITIAL, GOAL, ACTIONS)
if steps is None:
    print("No plan found")
else:
    print(f"Plan found, {len(steps)} actions:")
    for i, (s, a) in enumerate(zip(states, steps)):
        print(f"  S{i}: {sorted(s)}")
        print(f"       -- {a} -->")
    print(f"  S{len(steps)}: {sorted(states[-1])}")

Plan found, 4 actions:
  S0: ['At(Package,A)', 'At(Robot,A)']
       -- PickUp(Package,A) -->
  S1: ['At(Robot,A)', 'Holding(Package)']
       -- Move(A,B) -->
  S2: ['At(Robot,B)', 'Holding(Package)']
       -- Move(B,C) -->
  S3: ['At(Robot,C)', 'Holding(Package)']
       -- Drop(Package,C) -->
  S4: ['At(Package,C)', 'At(Robot,C)']


The generated plan matches the hand-worked one from Task 1 exactly (pick up at $A$, move
twice, drop at $C$) - four actions, which is also the minimum possible since BFS explores in
order of plan length.

## Task 3 - Testing the generated planner

**Test A - the original, solvable problem.**

In [5]:
assert steps is not None and len(steps) == 4
assert str(steps[0]).startswith("PickUp") and str(steps[-1]).startswith("Drop")
print("Test A passed: a 4-step plan was found and matches the manual trace")

Test A passed: a 4-step plan was found and matches the manual trace


**Test B - make the problem impossible** by removing every `PickUp` action, so the
package can never be lifted.

In [6]:
no_pickup = [a for a in ACTIONS if not a.name.startswith("PickUp")]
steps_b, _ = plan(INITIAL, GOAL, no_pickup)
print("Plan without PickUp actions:", steps_b)
assert steps_b is None
print("Test B passed: planner correctly reports no plan rather than inventing one")

Plan without PickUp actions: None
Test B passed: planner correctly reports no plan rather than inventing one


**Test C - an irrelevant action.** `Move(A,B)` can fire even while the package is still
at $A$; the planner must not confuse the robot reaching $C$ with the package reaching $C$.

In [7]:
# already true of the action set above, but check explicitly: a state where the
# robot alone has reached C, with the package left behind, must not satisfy the goal
robot_only_at_c = frozenset({"At(Robot,C)", "At(Package,A)"})
print("Does robot-only-at-C satisfy the goal?", GOAL <= robot_only_at_c)
assert not (GOAL <= robot_only_at_c)
print("Test C passed: reaching C with the robot alone is correctly not a goal state")

Does robot-only-at-C satisfy the goal? False
Test C passed: reaching C with the robot alone is correctly not a goal state


## Task 4 - Logic and search, side by side

| Specification idea | Where it is in the code |
|---|---|
| Preconditions -> is an action applicable? | `Action.applicable`, checking $S \models Preconditions(a)$ |
| Effects -> how does the state change? | `Action.apply`, computing $S' = Apply(S, a)$ |
| Goal -> when does planning stop? | `goal <= state` inside `plan` |
| BFS -> how are alternative plans explored? | the `deque` frontier in `plan`, expanded in FIFO order |

Filling in the diagram from the handout: *current state -> check preconditions ->
**apply the action's effects** -> generate successor state -> search over alternatives ->
goal?* The missing step is exactly what `Action.apply` does.

## Task 5 (optional) - Can the LLM verify its own plan?

Asking the same LLM to explain *why* the four-step plan is valid produces an explanation like:
"PickUp(Package,A) applies because the robot and package are both at A; this makes
Holding(Package) true, so the two Move actions are free to relocate the robot without
needing the package's location as a precondition; finally Drop(Package,C) applies because
the robot is now at C and still holding the package."

Comparing that explanation against the actual states printed by the Python program in
Task 2 word pointed to the same facts becoming true at the same points - the explanation
and the independently-computed trace agree here.

**Which should be trusted more: the explanation or the executed trace?** The executed
trace. The explanation is itself just more LLM output, generated after the fact, and an LLM
can narrate a plan confidently whether or not the underlying `apply`/`applicable` logic is
actually correct; the Python trace was produced by running the same `Action` objects used to
search for the plan, so an error in the logic would show up directly as a wrong state here,
whereas a wrong narrated explanation could still sound perfectly plausible.

## Reflection questions

1. **Why specify preconditions and effects before asking an LLM for the planner?** Without
   a precise specification there is nothing to check the generated code against - any plan
   the LLM invents would look equally "reasonable."
2. **Example error from a missing precondition check:** if `PickUp` did not require
   At(Robot, loc), the planner could pick up a package from across the warehouse without
   moving there first, producing a plan that looks valid on paper but cannot be executed by
   a real robot.
3. **Why isn't a "reasonable-looking" plan necessarily valid?** A plan can read correctly in
   English while silently skipping a precondition - the English gloss doesn't get checked by
   the logic, only the actual proposition sets do.
4. **What did the LLM contribute?** A working BFS-over-states implementation and the
   boilerplate for representing actions, saving the time of writing search bookkeeping by
   hand.
5. **What had to be verified independently?** That the four-step plan actually matched the
   hand-traced one from Task 1, and that the impossible/irrelevant-action tests behaved as
   expected - none of that is guaranteed just because the code runs without errors.
6. **Where is logical reasoning used here?** In `Action.applicable`, which checks whether
   $S \models Preconditions(a)$ before an action can fire.
7. **How does this connect to search?** Logic decides which actions are legal from a given
   state; search (BFS here) decides which sequence of legal actions to try, in order to find
   one that reaches the goal - exactly the "Logic + Search = Planning" idea from the
   handout.

## Optional extension: Prolog as an independent checker

A small Prolog knowledge base (`logic.pl`) encodes the warehouse connectivity as facts and
rules, and is queried from this notebook via `swipl`, completely independently of the Python
planner above.

In [8]:
%%writefile logic.pl
connected(a,b).
connected(b,a).
connected(b,c).
connected(c,b).

can_move(X,Y) :-
    connected(X,Y).

valid_move(X,Y) :-
    connected(X,Y).

wet_road.
slippery :- wet_road.
reduce_speed :- slippery.

Overwriting logic.pl


In [9]:
import subprocess

def ask_prolog(query, file="logic.pl"):
    goal = f"({query} -> writeln(true) ; writeln(false))"
    out = subprocess.run(
        ["swipl", "-q", "-s", file, "-g", goal, "-t", "halt"],
        capture_output=True, text=True,
    )
    return out.stdout.strip()

for q in ["can_move(a,b)", "can_move(a,c)"]:
    print(f"?- {q}.  -->  {ask_prolog(q)}")

?- can_move(a,b).  -->  true
?- can_move(a,c).  -->  false


`can_move(a,b)` succeeds because `connected(a,b)` is a fact. `can_move(a,c)` fails
because there is no `connected(a,c)` fact and no rule chains the two existing facts together
- Prolog only derives what logically follows from what is stated, it does not infer a path
of length two unless a rule says to look for one. The rule `can_move(X,Y) :- connected(X,Y)`
is just the direct logical implication $Connected(X,Y) \to CanMove(X,Y)$ written in Prolog
syntax.

### Checking the Python planner's moves with Prolog

The plan found above used `Move(A,B)` then `Move(B,C)`. Each step is checked against the
independent Prolog knowledge base via `valid_move/2`, and a move the planner never actually
proposed - `Move(a,c)` - is checked too.

In [10]:
for q in ["valid_move(a,b)", "valid_move(b,c)", "valid_move(a,c)"]:
    print(f"?- {q}.  -->  {ask_prolog(q)}")

?- valid_move(a,b).  -->  true
?- valid_move(b,c).  -->  true
?- valid_move(a,c).  -->  false


The first two queries succeed, matching the two `Move` actions actually used in the
Python plan. The third fails, confirming Prolog agrees that a direct $A \to C$ move is not
supported by the warehouse's connectivity - this is the same conclusion the Python search
reached indirectly (by never finding a single-action route between $A$ and $C$), now
confirmed by a completely separate reasoning system.

In [11]:
print("?- reduce_speed.  -->", ask_prolog("reduce_speed"))

?- reduce_speed.  --> true


**Explaining why `reduce_speed` succeeds:** Fact $\Rightarrow$ Rule $\Rightarrow$ Rule
$\Rightarrow$ Conclusion:

$$\text{wet\_road} \;\Rightarrow\; \text{slippery} \;\Rightarrow\; \text{reduce\_speed}.$$

`wet_road` is asserted as a fact; `slippery` follows from the rule `slippery :- wet_road`;
`reduce_speed` then follows from `reduce_speed :- slippery`. Prolog chains the two rules
together automatically when asked to prove `reduce_speed`.

### Prolog reflection

1. **Fact vs. rule:** a fact (e.g. `connected(a,b).`) is unconditionally true; a rule
   (e.g. `slippery :- wet_road.`) is only true when its body can itself be proven.
2. **Query vs. entailment:** asking `?- reduce_speed.` is exactly asking whether
   `reduce_speed` is entailed by the facts and rules in the knowledge base - Prolog answers
   `true`/`false` by trying to prove it.
3. **Why verify a Python-generated plan with Prolog?** The two systems share nothing in
   their implementation, so a bug specific to the Python search code (e.g. a typo in an
   effect) would not be reproduced by the Prolog check - agreement between them is much
   stronger evidence than the Python program simply not crashing.
4. **Advantage of an independent verifier when the plan itself came from an LLM-assisted
   program:** it removes the possibility that the same mistaken assumption caused both the
   plan and its "explanation" to look right - the verifier has to be convinced by the
   underlying facts, not by the planner's own account of itself.

**Key idea:** an AI system can propose a candidate solution, while a separate, simpler
logical system checks it independently.